# nb_setup_metadata
Run once (re-runnable). Creates the bronze/silver/gold schemas, `meta_entities` and `meta_run_log` in the **formula1** lakehouse.
Attach formula1 as the default lakehouse first. Landing layout assumed: `Files/landing/` with
`circuits.csv`, `races.csv`, `constructors.json`, `drivers.json`, and folders `results/` and `sprints/`.

In [1]:
from pyspark.sql.types import *

# ---------- medallion schemas (requires a schema-enabled lakehouse) ----------
for s in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")

# ---------- meta_entities: one row per source ----------
entities_schema = StructType([
    StructField("entity",       StringType(),  False),
    StructField("source_path",  StringType(),  False),   # file OR folder
    StructField("file_format",  StringType(),  False),   # csv | json
    StructField("delimiter",    StringType(),  True),    # csv only
    StructField("header",       BooleanType(), True),    # csv only
    StructField("multiline",    BooleanType(), True),    # json multi-line files
    StructField("null_value",   StringType(),  True),    # csv only; None = Spark default
    StructField("primary_key",  StringType(),  True),    # business key in SILVER column names, comma-separated
    StructField("bronze_table", StringType(),  False),
    StructField("write_mode",   StringType(),  False),   # overwrite | append
    StructField("load_type",    StringType(),  False),   # full | incremental (future)
    StructField("is_active",    IntegerType(), False),
])

LANDING = "Files/landing"

# (entity, path, format, delimiter, header, multiline, null_value, primary_key)
# Mirrors the original Databricks ingest notebooks.
sources = [
    ("circuits",     f"{LANDING}/circuits.csv",      "csv",  ",",  True,  None,  None, "circuit_id"),
    ("races",        f"{LANDING}/races.csv",         "csv",  ",",  True,  None,  None, "season,round"),
    ("constructors", f"{LANDING}/constructors.json", "json", None, None,  False, None, "constructor_id"),
    ("drivers",      f"{LANDING}/drivers.json",      "json", None, None,  False, None, "driver_id"),
    ("results",      f"{LANDING}/results",           "json", None, None,  False, None, "season,round,constructor_id,driver_id"),
    ("sprints",      f"{LANDING}/sprints",           "json", None, None,  True,  None, "season,round,constructor_id,driver_id"),
]

rows = [(e, p, f, d, h, m, n, pk, f"bronze.{e}", "overwrite", "full", 1)
        for (e, p, f, d, h, m, n, pk) in sources]

(spark.createDataFrame(rows, entities_schema)
      .write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable("meta_entities"))

# ---------- meta_run_log: observability ----------
spark.sql("""
CREATE TABLE IF NOT EXISTS meta_run_log (
    batch_id     STRING,
    entity       STRING,
    layer        STRING,
    status       STRING,
    rows_written BIGINT,
    start_ts     TIMESTAMP,
    end_ts       TIMESTAMP,
    error_msg    STRING
) USING DELTA
""")

display(spark.table("meta_entities"))

StatementMeta(, 0f854e60-9b9d-4b24-86e1-eb2d4aaf70f2, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f99c7c2-939d-405a-b5e5-52acac0cd247)

# nb_seed_meta_columns
Seeds `meta_columns` (one row per bronze column) by profiling the bronze tables.
Types are *guessed* from the data (int, double, date, timestamp, else string) and names are converted to snake_case.
**Review and edit the table afterwards** - it is the source of truth for the silver notebook.
Re-running only adds entities that are not already in `meta_columns`, so your edits are never overwritten.